# RAG Agent Evaluation Pipeline — Kaggle 2×T4

## Why this notebook exists (context for future reference)

The original plan was to run RAGAS evaluation **locally** using `.py` scripts
(`categorize.py`, `metrics.py`, `report.py`) with **Groq's `gpt-oss-120b`** as the LLM judge.

That plan failed for two compounding reasons:

1. **RAGAS fires all judge calls concurrently.** With 3 metrics × ~3 LLM calls each × 147 samples,
   this creates ~1,300+ API calls that immediately saturate any free-tier provider
   (Groq free tier: 30 RPM, 8K–12K TPM depending on model). `RunConfig(max_workers=1)` slows
   RAGAS down but does not fully serialize Groq-side — bursts still trigger 429s.

2. **Payload Too Large (HTTP 413).** RAGAS sends full retrieved chunks inside faithfulness and
   context-recall prompts. Multi-hop samples carry 4–6 large chunks. A single judge call easily
   exceeds 10K tokens — far beyond any free-tier TPM cap. Truncating chunks would directly
   corrupt the faithfulness metric (the judge can't verify grounding without the full context).

**Solution:** Run the judge LLM locally on Kaggle's free 2×T4 GPUs (32 GB VRAM) via Ollama.
No rate limits, no payload size constraints, full chunks passed to the judge every time.

## What this notebook does

```
results.jsonl          ← collected locally by runner.py (agent responses)
golden_dataset.json    ← 147 QA pairs from eval-data-generation.ipynb
        │
        ├─ [Step 3] Categorize into 3 buckets
        │           retrieval_samples  → agent used retrieval (Path A / B)
        │           unflagged_samples  → no retrieval, gave a real answer
        │           flagged_samples    → no retrieval, refused / errored → score=0
        │
        ├─ [Step 4] Install Ollama + pull judge model + set num_ctx via Modelfile
        │           Load BGE-M3 embeddings on CPU
        │
        ├─ [Step 5] RAGAS evaluation per bucket (1 sample at a time)
        │           retrieval  → Faithfulness + ContextRecall + AnswerCorrectness
        │           unflagged  → AnswerCorrectness only
        │           flagged    → score=0, no LLM call
        │
        └─ [Step 6] Build + print final report → /kaggle/working/eval_report.json
```

## GPU layout
| Component | Runs on | Memory |
|---|---|---|
| Judge LLM (mistral-small3.2 24B Q4) | Both T4s via Ollama auto-split | ~14–15 GB VRAM |
| BGE-M3 embeddings | CPU | ~2 GB RAM |

## num_ctx — why it matters
Ollama defaults to 4096 context tokens. RAGAS faithfulness prompts include the full retrieved
context + question + answer. From our chunk size distribution, parent chunks can reach 40K+
characters. Even typical multi-hop samples (4–6 chunks) can push past 10K tokens per judge call.
We set `num_ctx=32768` via a Modelfile so Ollama never silently truncates sequences.

## Why `LangchainLLMWrapper` + `response_format: json_object`

`LangchainLLMWrapper` is used here (not `LiteLLMStructuredLLM`) because `ragas==0.4.3`'s
`ragas.metrics` (legacy namespace) requires a `LangchainLLMWrapper` — the newer
`ragas.metrics.collections` namespace requires `InstructorBaseRagasLLM` but is incompatible
with our Ollama setup. Setting `response_format: {"type": "json_object"}` forces Ollama to
return structured JSON on every call, which is sufficient to prevent `OutputParserException`
in this version of RAGAS.

## Step 0 — Verify 2×T4 GPUs

Make sure the accelerator is set to **GPU T4 x2** in Kaggle Settings before running.

In [1]:
import os, subprocess

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

result = subprocess.run(
    ['nvidia-smi', '--query-gpu=index,name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
)
assert result.returncode == 0, "No GPU found. Enable: Kaggle Settings → Accelerator → GPU T4 x2"

gpus = [g.strip() for g in result.stdout.strip().split('\n')]
print(f"Found {len(gpus)} GPU(s):")
for g in gpus: print(" ", g)

assert len(gpus) >= 2, f"Need 2 GPUs, found {len(gpus)}. Change accelerator to 'GPU T4 x2'."

Found 2 GPU(s):
  0, Tesla T4, 15360 MiB
  1, Tesla T4, 15360 MiB


## Step 1 — Install Dependencies

- `ragas==0.4.3` — pinned to match the data-generation notebook.
- `langchain-huggingface` + `sentence-transformers` — for BGE-M3 embeddings on CPU
- `rapidfuzz` — used internally by RAGAS for fuzzy matching
- `openai` + `instructor` — pre-installed on Kaggle; Ollama exposes an OpenAI-compatible
  endpoint that LiteLLM uses under the hood

In [2]:
!pip install -q "ragas==0.4.3" langchain-huggingface sentence-transformers rapidfuzz

import openai, instructor
print(f"openai {openai.__version__} | instructor {instructor.__version__}")
print("✅ Done.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 10.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 58.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.5/252.5 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 64.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.1/122.1 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 87.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.3 MB/s eta 0:00:00
ERROR: pip's dependency reso

## Step 1b — Patch broken RAGAS legacy imports

`ragas==0.4.3` has stale imports for `langchain_community.chat_models.vertexai` and
`langchain_community.llms.vertexai` that no longer exist in the installed version of
`langchain-community`. Injecting empty stub modules lets RAGAS import cleanly.

In [3]:
import sys, types

for mod_path in ["langchain_community.chat_models.vertexai",
                 "langchain_community.llms.vertexai"]:
    stub = types.ModuleType(mod_path)
    stub.ChatVertexAI = type("ChatVertexAI", (), {})
    stub.VertexAI     = type("VertexAI",     (), {})
    sys.modules[mod_path] = stub

import ragas
print(f"✅ RAGAS {ragas.__version__} imported successfully.")

✅ RAGAS 0.4.3 imported successfully.


## Step 2 — Upload Input Files

Run this cell to upload both files interactively.
Files needed:
- `results.jsonl`      — agent responses (from `runner.py` locally)
- `golden_dataset.json` — reference QA pairs (from data generation notebook)

In [4]:
# from kaggle_secrets import UserSecretsClient
from IPython.display import display
import ipywidgets as widgets
import json, shutil
from pathlib import Path

DATA_DIR = Path("/kaggle/working/data")
DATA_DIR.mkdir(exist_ok=True)

RESULTS_PATH = DATA_DIR / "results.jsonl"
DATASET_PATH = DATA_DIR / "golden_dataset.json"

# ── Upload via Kaggle file upload widget ─────────────────────────────────────
# If you already have the files in /kaggle/input/, copy them instead:
# shutil.copy("/kaggle/input/<your-dataset>/results.jsonl", RESULTS_PATH)
# shutil.copy("/kaggle/input/<your-dataset>/golden_dataset.json", DATASET_PATH)

upload = widgets.FileUpload(accept='.jsonl,.json', multiple=True, description='Upload files')
display(upload)
print("Select BOTH results.jsonl and golden_dataset.json, then run the next cell.")

FileUpload(value=(), accept='.jsonl,.json', description='Upload files', multiple=True)

Select BOTH results.jsonl and golden_dataset.json, then run the next cell.


In [5]:
# Save uploaded files to /kaggle/working/data/
for file_info in upload.value:
    name = file_info['name']
    content = file_info['content']
    dest = DATA_DIR / name
    with open(dest, 'wb') as f:
        f.write(content)
    print(f"Saved: {dest} ({len(content)/1024:.1f} KB)")

# Verify both files exist
assert RESULTS_PATH.exists(),  f"Missing: {RESULTS_PATH}"
assert DATASET_PATH.exists(),  f"Missing: {DATASET_PATH}"

# Quick sanity check on contents
with open(RESULTS_PATH)  as f: results_count  = sum(1 for _ in f)
with open(DATASET_PATH)  as f: dataset_count  = len(json.load(f))
print(f"results.jsonl      : {results_count} rows")
print(f"golden_dataset.json: {dataset_count} entries")

# If resuming from a crash, also copy the checkpoint file
ckpt_upload = DATA_DIR / "retrieval_scores_checkpoint.jsonl"
if ckpt_upload.exists():
    import shutil
    shutil.copy(ckpt_upload, "/kaggle/working/retrieval_scores_checkpoint.jsonl")
    print(f"Checkpoint restored: {ckpt_upload}")

Saved: /kaggle/working/data/results.jsonl (3104.6 KB)
Saved: /kaggle/working/data/golden_dataset.json (1134.0 KB)
results.jsonl      : 145 rows
golden_dataset.json: 147 entries


## Step 3 — Categorize Results

Mirrors `categorize.py` from the local pipeline. Splits results into three buckets:

| Bucket | Condition | Eval treatment |
|---|---|---|
| `retrieval_samples` | `retrieval_performed=True` | Faithfulness + ContextRecall + AnswerCorrectness |
| `unflagged_samples` | no retrieval, real answer | AnswerCorrectness only |
| `flagged_samples` | no retrieval, refused / errored | score=0, no LLM call |

In [6]:
# Refusal/hedge phrases from prompts.py — extend here if prompts change
REFUSAL_PHRASES = [
    "cannot produce results",
    "only indexes the official",
    "please refer to the official api",
    "please ask a clear question",
    "try rephrasing",
    "retrieval insights do not provide",
    "insufficient documentation",
    "i cannot find",
    "please rephrase",
    "api inferences are out of scope",
]

def is_refusal(response: str) -> bool:
    lowered = response.lower()
    return any(phrase in lowered for phrase in REFUSAL_PHRASES)


def categorize(results: list) -> tuple:
    retrieval_samples, unflagged_samples, flagged_samples = [], [], []
    for r in results:
        # Agent crashes (network error, etc.) always go to flagged
        if r.get('error') or not r.get('agent_response'):
            r['flag_reason'] = 'agent_error'
            flagged_samples.append(r)
        elif r['retrieval_performed']:
            retrieval_samples.append(r)
        elif is_refusal(r['agent_response']):
            r['flag_reason'] = 'refusal_detected'
            flagged_samples.append(r)
        else:
            unflagged_samples.append(r)
    return retrieval_samples, unflagged_samples, flagged_samples


# Load results
results = []
with open(RESULTS_PATH, encoding='utf-8') as f:
    for line in f:
        results.append(json.loads(line))

retrieval_samples, unflagged_samples, flagged_samples = categorize(results)

print("Categorization summary")
print(f"  Retrieval performed    : {len(retrieval_samples)}")
print(f"  No retrieval — clean   : {len(unflagged_samples)}")
print(f"  No retrieval — flagged : {len(flagged_samples)}")
print(f"  Total                  : {len(results)}")

if flagged_samples:
    print("\nFlagged samples:")
    for r in flagged_samples:
        print(f"  [{r.get('flag_reason')}] {r['user_input'][:80]}")

Categorization summary
  Retrieval performed    : 117
  No retrieval — clean   : 4
  No retrieval — flagged : 24
  Total                  : 145

Flagged samples:
  [agent_error] Whate ar langchain integrtions with Azure?
  [refusal_detected] wat is bert in SAP HANA Cloud data graph
  [refusal_detected] What benefits does Amazon AWS Lambda provide for developers in the langchain fra
  [refusal_detected] What kind of stats can you get from a vector_store in the ZeusDB?
  [refusal_detected] How does the Kanon 2 Embedder contribute to the embedding process in Isaacus's l
  [refusal_detected] What is the role description for a Software Engineer (Mobile App Development) at
  [refusal_detected] Why is the Higgs Boson important?
  [refusal_detected] Cn u explain hw LangGraph integr8s wi8h CopilotKit to provode structurd UI paylo
  [refusal_detected] Can yu tell me wat does Daniel | Tech & Data say bout bypassing Cloudflare while
  [refusal_detected] How does President Biden mention Beau Biden 

## Step 4 — Install Ollama, Pull Judge Model, Load BGE-M3

### Judge model options (change `JUDGE_MODEL_BASE` below to switch)
| Model | VRAM Q4 | Notes |
|---|---|---|
| `mistral-small3.2` | ~14–15 GB | Comfortable headroom, strong reasoning |
| `command-r:35b` | ~20–21 GB | Purpose-built for RAG, tighter VRAM margin on 2×T4 ✅ active| 

### Setting num_ctx via Modelfile
Ollama's `--parameter` flag on `pull` does not persist context size. The only reliable
method is to create a Modelfile that derives from the pulled base and sets `num_ctx`
permanently, then register it as a named model with `ollama create`.

In [17]:
import subprocess, time, urllib.request, os, shutil

# Set Ollama env vars BEFORE starting the server
os.environ["OLLAMA_NUM_PARALLEL"]      = "1"
os.environ["OLLAMA_MAX_LOADED_MODELS"] = "1"
os.environ["OLLAMA_KEEP_ALIVE"]        = "24h"

# zstd is required by Ollama's installer on Kaggle
print("Installing zstd...")
subprocess.run("apt-get update -qq && apt-get install -y -qq zstd", shell=True, check=True)

# Remove stale binary if present from a previous run
if os.path.exists("/usr/local/bin/ollama"):
    os.remove("/usr/local/bin/ollama")

# No check=True — installer exits non-zero on Kaggle due to systemd, even on success
print("Installing Ollama...")
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True)

if not (shutil.which("ollama") or os.path.exists("/usr/local/bin/ollama")):
    raise RuntimeError("Ollama binary not found after install. Check logs above.")
print("✅ Ollama installed.")

# Start server in background
ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

Installing zstd...


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


Installing Ollama...


>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
#######################################################################   99.3%

✅ Ollama installed.


######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [18]:
# Poll until server is accepting connections
print("Waiting for Ollama server", end="")
for _ in range(30):
    time.sleep(2)
    try:
        urllib.request.urlopen("http://localhost:11434/", timeout=2)
        print("\n✅ Server ready.")
        break
    except Exception:
        print(".", end="", flush=True)
else:
    raise RuntimeError("Ollama server failed to start in 60 seconds.")

Waiting for Ollama server.
✅ Server ready.


In [9]:
# ── Judge model selection — comment/uncomment to switch ──────────────────────
# JUDGE_MODEL = "mistral-small3.2"   # 24B Q4_K_M, ~14 GB VRAM — active
JUDGE_MODEL = "command-r:35b"    # 35B Q4_K_M, ~22 GB VRAM — uncomment to switch

JUDGE_MODEL_EXTENDED = f"{JUDGE_MODEL}-eval"  # name for our num_ctx-extended variant

print(f"Pulling {JUDGE_MODEL} (~22 GB, takes 10-15 min on first run)...")
subprocess.run(["ollama", "pull", JUDGE_MODEL],
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print("Base model pulled.")

# Modelfile extends the base with larger context window.
# 32768 chosen over 32768 because multi-hop samples with 4-6 large chunks
# can exceed 16K tokens in the RAGAS faithfulness prompt.
modelfile_content = f"FROM {JUDGE_MODEL}\nPARAMETER num_ctx 32768\n"
modelfile_path = "/tmp/Modelfile_eval"
with open(modelfile_path, "w") as f:
    f.write(modelfile_content)

subprocess.run(["ollama", "create", JUDGE_MODEL_EXTENDED, "-f", modelfile_path], check=True)
print(f"Extended model '{JUDGE_MODEL_EXTENDED}' created with num_ctx=32768.")

# Quick smoke test — confirms model loads and responds before the long eval run
test = subprocess.run(
    ["ollama", "run", JUDGE_MODEL_EXTENDED, "Reply with: OK"],
    capture_output=True, text=True, timeout=300
)
print(f"Model test: {test.stdout.strip()[:50]}")
print("✅ Judge LLM ready.")

Pulling command-r:35b (~22 GB, takes 10-15 min on first run)...
Base model pulled.
Extended model 'command-r:35b-eval' created with num_ctx=32768.


gathering model components 
using existing layer sha256:8e0609b8f0fe36530ac9966544e6cf780cfcda006fbe739eeab278aef89b3df7 
using existing layer sha256:b3741b7b9ce59bd791b1e58f10b80144ff2345e81345723d52f0c68bc61eb46e 
using existing layer sha256:922095537bc1278418b3aeff5c9bdde0c61a5c6adb7573498b09791a95044068 
using existing layer sha256:945eaa8b14280408800a409b7db5d18dbb266a21610d85cb8eb32c93c3243f90 
creating new layer sha256:4d0d31d037af6de85d4fe7d6615e45f27a194ca06b16f830694064d9181dca17 
writing manifest 
success 


Model test: OK
✅ Judge LLM ready.


In [10]:
from langchain_openai import ChatOpenAI
from ragas.llms import LangchainLLMWrapper

# Wire judge to Ollama's OpenAI-compatible endpoint.
# response_format json_object forces structured output — reduces OutputParserExceptions
# that LangchainLLMWrapper's parse_output_string can trigger on plain-text responses.
judge_llm = LangchainLLMWrapper(
    ChatOpenAI(
        base_url     = "http://localhost:11434/v1",
        api_key      = "ollama",
        model        = JUDGE_MODEL_EXTENDED,
        temperature  = 0,
        model_kwargs = {"response_format": {"type": "json_object"}},
    )
)
print(f"✅ Judge LLM ready: {JUDGE_MODEL_EXTENDED}")

✅ Judge LLM ready: command-r:35b-eval


/tmp/ipykernel_58/533619972.py:7: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  judge_llm = LangchainLLMWrapper(


In [11]:
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_huggingface import HuggingFaceEmbeddings

print("Loading BAAI/bge-m3 on CPU (first run downloads ~570 MB)...")
bge_embeddings = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(
        model_name   = "BAAI/bge-m3",
        model_kwargs = {"device": "cpu"},
    )
)
print("✅ BGE-M3 ready.")

Loading BAAI/bge-m3 on CPU (first run downloads ~570 MB)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

✅ BGE-M3 ready.


/tmp/ipykernel_58/4133884220.py:5: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  bge_embeddings = LangchainEmbeddingsWrapper(


## Step 5 — Wire RAGAS to Ollama and Run Metrics

We use `LiteLLMStructuredLLM` (an `InstructorBaseRagasLLM` subclass) instead of
`LangchainLLMWrapper` — see the title cell for the full explanation.

### Extracting scores from `EvaluationResult`
In `ragas==0.4.3`, `dict(result)` attempts integer indexing and raises `KeyError: 0`.
The correct extraction is `result.to_pandas()` which returns a DataFrame with one row
per sample and one column per metric. We take `.mean()` to get aggregate scores.

In [14]:
# from ragas.metrics.collections import Faithfulness, ContextRecall, AnswerCorrectness
from ragas.metrics import Faithfulness, ContextRecall, AnswerCorrectness
from ragas import SingleTurnSample, EvaluationDataset, evaluate
from ragas.run_config import RunConfig

# Single worker + long timeout — Ollama is a single-server process,
# concurrent requests queue anyway and add overhead.
# 1200s covers worst-case: faithfulness on a 6-chunk multi-hop sample.
run_cfg = RunConfig(max_workers=1, timeout=1200)

print("✅ Metrics and run config ready.")

✅ Metrics and run config ready.


/tmp/ipykernel_58/483827792.py:2: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, ContextRecall, AnswerCorrectness
/tmp/ipykernel_58/483827792.py:2: DeprecationWarning: Importing ContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextRecall
  from ragas.metrics import Faithfulness, ContextRecall, AnswerCorrectness
/tmp/ipykernel_58/483827792.py:2: DeprecationWarning: Importing AnswerCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerCorrectness
  from ragas.metrics import Faithfulness, ContextRecall, AnswerCorrectness


In [19]:
import json
from pathlib import Path

CHECKPOINT_PATH = Path("/kaggle/working/retrieval_scores_checkpoint.jsonl")

def load_completed_indices() -> set:
    """Return indices of samples already scored in the checkpoint file."""
    if not CHECKPOINT_PATH.exists():
        return set()
    completed = set()
    with open(CHECKPOINT_PATH) as f:
        for line in f:
            completed.add(json.loads(line)["index"])
    return completed


def eval_single_retrieval_sample(r: dict) -> dict:
    """Run RAGAS on one sample and return the metric scores."""
    dataset = EvaluationDataset(samples=[
        SingleTurnSample(
            user_input         = r["user_input"],
            response           = r["agent_response"],
            retrieved_contexts = r["retrieved_contexts"],
            reference          = r["reference"],
        )
    ])
    result = evaluate(
        dataset    = dataset,
        metrics    = [
            Faithfulness(llm=judge_llm),
            ContextRecall(llm=judge_llm),
            AnswerCorrectness(llm=judge_llm, embeddings=bge_embeddings),
        ],
        run_config = run_cfg,
    )
    row = result.to_pandas().iloc[0]
    return {
        "faithfulness"      : float(row["faithfulness"]),
        "context_recall"    : float(row["context_recall"]),
        "answer_correctness": float(row["answer_correctness"]),
    }


retrieval_scores = {}

if not retrieval_samples:
    print("No retrieval samples — skipping.")
else:
    completed = load_completed_indices()
    print(f"Retrieval samples: {len(retrieval_samples)} total | "
          f"{len(completed)} already scored | "
          f"{len(retrieval_samples) - len(completed)} remaining\n")

    with open(CHECKPOINT_PATH, "a") as ckpt:
        for i, r in enumerate(retrieval_samples):
            if i in completed:
                continue

            print(f"[{i+1}/{len(retrieval_samples)}] {r['user_input'][:70]}...")
            try:
                scores = eval_single_retrieval_sample(r)
                record = {"index": i, "user_input": r["user_input"],
                          "question_type": r.get("question_type", ""), **scores}
                ckpt.write(json.dumps(record) + "\n")
                ckpt.flush()  # write immediately — don't buffer
                print(f"  faithfulness={scores['faithfulness']:.3f} | "
                      f"context_recall={scores['context_recall']:.3f} | "
                      f"answer_correctness={scores['answer_correctness']:.3f}")
            except Exception as e:
                print(f"  ⚠️ Failed: {e} — skipping, will retry on next run")

    # Aggregate scores from checkpoint file for the report
    all_records = []
    with open(CHECKPOINT_PATH) as f:
        for line in f:
            all_records.append(json.loads(line))

    import pandas as pd
    scores_df = pd.DataFrame(all_records)
    retrieval_scores = scores_df[["faithfulness", "context_recall", "answer_correctness"]].mean().to_dict()

    print(f"\nRetrieval scores ({len(all_records)} samples scored):")
    for metric, score in retrieval_scores.items():
        print(f"  {metric:<30} {score:.4f}")

Retrieval samples: 117 total | 0 already scored | 117 remaining

[1/117] As a Tech Contributor to the langchain framework, how should I use Git...


Evaluating:   0%|          | 0/3 [00:00<?, ?it/s]

  faithfulness=1.000 | context_recall=0.500 | answer_correctness=0.399
[2/117] How can openai:gpt-5.5 be integrated using LangChain in a web applicat...


Evaluating:   0%|          | 0/3 [00:00<?, ?it/s]

  faithfulness=0.800 | context_recall=0.000 | answer_correctness=0.469
[3/117] How can short-term memory be accessed in a tool within the langchain f...


...

  faithfulness=0.667 | context_recall=1.000 | answer_correctness=0.620
[116/117] How does the requirement for TLS relate to handling high read and writ...


Evaluating:   0%|          | 0/3 [00:00<?, ?it/s]

  faithfulness=0.636 | context_recall=1.000 | answer_correctness=0.669
[117/117] How does error handling in the LangGraph framework relate to assistant...


Evaluating:   0%|          | 0/3 [00:00<?, ?it/s]

  faithfulness=1.000 | context_recall=0.667 | answer_correctness=0.720

Retrieval scores (119 samples scored):
  faithfulness                   0.7887
  context_recall                 0.7694
  answer_correctness             0.5304


In [20]:
unflagged_scores = {}

if not unflagged_samples:
    print("No unflagged samples — skipping.")
else:
    print(f"Evaluating {len(unflagged_samples)} unflagged (no-retrieval) samples...")
    print("Metric: AnswerCorrectness only (no retrieved context to judge)\n")

    dataset = EvaluationDataset(samples=[
        SingleTurnSample(
            user_input = r["user_input"],
            response   = r["agent_response"],
            reference  = r["reference"],
        )
        for r in unflagged_samples
    ])

    result = evaluate(
        dataset    = dataset,
        metrics    = [AnswerCorrectness(llm=judge_llm, embeddings=bge_embeddings)],
        run_config = run_cfg,
    )

    scores_df = result.to_pandas()
    unflagged_scores = scores_df[['answer_correctness']].mean().to_dict()

    print("Unflagged scores:")
    for metric, score in unflagged_scores.items():
        print(f"  {metric:<30} {score:.4f}")

# Flagged samples get score=0 — no LLM call needed
flagged_scores = {
    "count"             : len(flagged_samples),
    "answer_correctness": 0.0,
    "note"              : "All flagged samples assigned score=0 (refusal or agent error)",
}
print(f"\nFlagged samples: {len(flagged_samples)} → score=0")

Evaluating 4 unflagged (no-retrieval) samples...
Metric: AnswerCorrectness only (no retrieved context to judge)



Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

Unflagged scores:
  answer_correctness             0.4077

Flagged samples: 24 → score=0


## Step 6 — Build Report

Three eval views, each sliced by `question_type` (single-hop vs multi-hop):

1. **Retrieval performed** — full metric suite
2. **No-retrieval combined** — unflagged + flagged, flagged counted as 0 in weighted average
3. **No-retrieval unflagged only** — clean subset without refused/errored samples

In [21]:
from collections import defaultdict

def question_type_label(qtype: str) -> str:
    """Normalize RAGAS synthesizer names to simple hop labels."""
    if 'single' in qtype: return 'single_hop'
    if 'multi'  in qtype: return 'multi_hop'
    return 'unknown'

def count_by_question_type(samples: list) -> dict:
    counts = defaultdict(int)
    for r in samples:
        counts[question_type_label(r.get('question_type', ''))] += 1
    return dict(counts)


# Weighted answer_correctness for the combined no-retrieval view.
# Flagged samples contribute 0 proportionally.
total_no_retrieval = len(unflagged_samples) + len(flagged_samples)
if total_no_retrieval > 0 and unflagged_scores:
    weight               = len(unflagged_samples) / total_no_retrieval
    combined_correctness = unflagged_scores.get('answer_correctness', 0.0) * weight
else:
    combined_correctness = 0.0


report = {
    "retrieval_performed": {
        "sample_count"     : len(retrieval_samples),
        "scores"           : retrieval_scores,
        "by_question_type" : count_by_question_type(retrieval_samples),
    },
    "no_retrieval_combined": {
        "sample_count"      : total_no_retrieval,
        "flagged_count"     : len(flagged_samples),
        "unflagged_count"   : len(unflagged_samples),
        "answer_correctness": round(combined_correctness, 4),
        "by_question_type"  : count_by_question_type(unflagged_samples + flagged_samples),
    },
    "no_retrieval_unflagged_only": {
        "sample_count"     : len(unflagged_samples),
        "scores"           : unflagged_scores,
        "by_question_type" : count_by_question_type(unflagged_samples),
    },
}


# ── Print readable summary ────────────────────────────────────────────────────
print("=" * 60)
print("EVAL REPORT")
print("=" * 60)

sec = report['retrieval_performed']
print(f"\n[1] Retrieval performed  (n={sec['sample_count']})")
for metric, score in sec.get('scores', {}).items():
    print(f"    {metric:<30} {score:.4f}")
for qtype, count in sec.get('by_question_type', {}).items():
    print(f"    └─ {qtype}: {count} samples")

sec = report['no_retrieval_combined']
print(f"\n[2] No-retrieval — combined  (n={sec['sample_count']})")
print(f"    flagged (score=0)  : {sec['flagged_count']}")
print(f"    unflagged          : {sec['unflagged_count']}")
print(f"    answer_correctness (weighted): {sec['answer_correctness']:.4f}")
for qtype, count in sec.get('by_question_type', {}).items():
    print(f"    └─ {qtype}: {count} samples")

sec = report['no_retrieval_unflagged_only']
print(f"\n[3] No-retrieval — unflagged only  (n={sec['sample_count']})")
for metric, score in sec.get('scores', {}).items():
    print(f"    {metric:<30} {score:.4f}")
for qtype, count in sec.get('by_question_type', {}).items():
    print(f"    └─ {qtype}: {count} samples")

print("\n" + "=" * 60)


# ── Save to Kaggle output ─────────────────────────────────────────────────────
REPORT_PATH = "/kaggle/working/eval_report.json"
with open(REPORT_PATH, 'w') as f:
    json.dump(report, f, indent=2)

print(f"\n✅ Report saved to {REPORT_PATH}")
print("   Download: Kaggle notebook → Output tab → eval_report.json")

EVAL REPORT

[1] Retrieval performed  (n=117)
    faithfulness                   0.7887
    context_recall                 0.7694
    answer_correctness             0.5304
    └─ single_hop: 64 samples
    └─ multi_hop: 53 samples

[2] No-retrieval — combined  (n=28)
    flagged (score=0)  : 24
    unflagged          : 4
    answer_correctness (weighted): 0.0582
    └─ single_hop: 26 samples
    └─ multi_hop: 2 samples

[3] No-retrieval — unflagged only  (n=4)
    answer_correctness             0.4077
    └─ single_hop: 4 samples


✅ Report saved to /kaggle/working/eval_report.json
   Download: Kaggle notebook → Output tab → eval_report.json


## Step 7 — Cleanup

Stop the Ollama server to release GPU VRAM.

In [22]:
ollama_proc.terminate()
print("✅ Ollama server stopped.")

✅ Ollama server stopped.
